# Ebook RAG con LlamaIndex

Inspirado en [huggingface](https://huggingface.co/learn/cookbook/en/rag_llamaindex_librarian)


## Dependencias

Para correr los modelos de manera local.


In [ ]:
%pip install -q \
    llama-index \
    EbookLib \
    html2text \
    llama-index-embeddings-huggingface \
    llama-index-llms-ollama \
    llama-index-readers-file

Instalar Ollama de https://ollama.com/download/OllamaSetup.exe


In [1]:
import subprocess, time, urllib.request

def ollama_is_up(timeout: float = 1.0) -> bool:
    try:
        urllib.request.urlopen("http://localhost:11434/api/tags", timeout=timeout)
        return True
    except Exception:
        return False

if not ollama_is_up():
    print("Starting Ollama server...")
    subprocess.Popen(
        ["ollama", "serve"],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
        # hides the console window on Windows; harmless on other OSes
        creationflags=getattr(subprocess, "CREATE_NO_WINDOW", 0),
    )
    for _ in range(60):
        if ollama_is_up():
            print("Ollama is up.")
            break
        time.sleep(1)
    else:
        raise RuntimeError("Ollama did not become ready in 60s.")
else:
    print("Ollama is already running.")

Starting Ollama server...
Ollama is up.


Pull Llama2 from the Ollama library.


In [ ]:
!ollama pull llama2

## Documentos

Descargamos epubs de la pagina [Project Gutenberg](https://www.gutenberg.org/ebooks/subject/10163), bajo la tematica "Filosofia de la Religion" (que esten en ingles).


In [ ]:
from pathlib import Path
import urllib.request
import shutil

library_dir = Path("./test/library")
library_dir.mkdir(parents=True, exist_ok=True)

books = {
    "elementary_forms_religious_life-Durkheim.epub":         "https://www.gutenberg.org/ebooks/41360.epub.noimages",
    "essence_christianity-Feuerbach.epub":                   "https://www.gutenberg.org/ebooks/47025.epub.noimages",
    "religious_sentiment-Brinton.epub":                      "https://www.gutenberg.org/ebooks/30061.epub.noimages",
    "philosophy_natural_theology-Jackson.epub":              "https://www.gutenberg.org/ebooks/47314.epub.noimages",
    "myth_science-Vignoli.epub":                             "https://www.gutenberg.org/ebooks/17802.epub.noimages",
    "authorize_bible_interpeters-Beecher.epub":              "https://www.gutenberg.org/ebooks/56354.epub.noimages",
    "sources_religious_insight-Royce.epub":                  "https://www.gutenberg.org/ebooks/33677.epub.noimages",
    "philosophy_religion_psychology_history-Sabatier.epub":  "https://www.gutenberg.org/ebooks/38446.epub.noimages",
    "god_world-Robinson.epub":                               "https://www.gutenberg.org/ebooks/30709.epub.noimages",
    "common_sense_religion-Beecher.epub":                    "https://www.gutenberg.org/ebooks/55531.epub.noimages",
}

for name, url in books.items():
    dest = library_dir / name
    if dest.exists() and dest.stat().st_size > 0:
        print(f"[skip] {name}")
        continue
    print(f"[dl]   {name}")
    req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
    with urllib.request.urlopen(req, timeout=60) as resp, open(dest, "wb") as f:
        shutil.copyfileobj(resp, f)

print("Done.")

## LlamaIndex

RAG with LlamaIndex, at its core, consists of the following broad phases:

1. **Loading**, de los documentos;
2. **Indexing**, vector embeddings;
3. **Querying**, para recuper los documentos relevantes.


### Loading

Naturally, let's start with the **loading** phase.

El objeto
[`SimpleDirectoryReader`](https://docs.llamaindex.ai/en/stable/module_guides/loading/simpledirectoryreader.html)
soporta archivos en epub.


In [2]:
from llama_index.core import SimpleDirectoryReader
from llama_index.readers.file import EpubReader
import random

loader = SimpleDirectoryReader(
    input_dir="./test/",
    recursive=True,
    required_exts=[".epub"],
    file_extractor={".epub": EpubReader()},
)

documents = loader.load_data()

len(documents)

10

In [3]:
for doc in random.choices(documents,k=2):  # Check the first couple of documents
    print("="*20)
    txt = doc.text
    n = len(txt)
    L = 1000
    a = random.randint(101, n-L-1)
    print(txt[:100])
    print(txt[a:a+L])

## The Project Gutenberg eBook of Myth and Science

This eBook is for the use of anyone anywhere in 
 memory, with all the manifold resemblances, as well as the ideal
of certain types in which the numerous forms we have seen and compared are
formulated. We know that when the memory has been dormant, which is often the
case, it may be awakened by the stimulus of association, of analogy, or of
will, so as to reproduce the forgotten ideas and sensations which are thus
again presented to the consciousness. When, therefore, one or more objects are
seen in an uncertain light, so as to present a confused appearance of the
human form, its general lineaments are unconsciously made by us to correspond
with the human type already existing in the memory, and this type presides in
the subsequent composition of the reproducing artist who observes the
phantasm. The unconscious mental labour which is accomplished in the
reproducing cellules of past impressions and ideas by the instantaneous
creation o

### Indexing

Se usa
[`BAAI/bge-small-en-v1.5`](https://huggingface.co/BAAI/bge-base-en-v1.5) to
para generar los embeddings.


In [4]:
from llama_index.embeddings.huggingface import HuggingFaceEmbedding

embedding_model = HuggingFaceEmbedding(model_name="BAAI/bge-small-en-v1.5")

c:\Users\DELL\Documents\Areas\MCD\3 S\PLN\RAG-Testing\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-11-15 20:55:36,531 - INFO - HTTP Request: GET https://huggingface.co/api/agent-harnesses "HTTP/1.1 200 OK"
2026-11-15 20:55:36,658 - INFO - HTTP Request: GET https://huggingface.co/api/models/BAAI/bge-small-en-v1.5 "HTTP/1.1 200 OK"
2026-11-15 20:55:36,893 - INFO - Loading SentenceTransformer model from BAAI/bge-small-en-v1.5.
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 349.89it/s]
2026-11-15 20:55:43,557 - INFO - HTTP Request: GET https://huggingface.co/api/models/BAAI/bge-small-en-v1.5/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"
2026-11-15 20:55:43,675 - INFO - HTTP Request: GET https://huggingface.co/api/models/BAAI/bge-small-en-v1.5/tree/main

In [5]:
from llama_index.core import VectorStoreIndex

index = VectorStoreIndex.from_documents(
    documents,
    embed_model=embedding_model,
)

In [7]:
retriever = index.as_retriever(similarity_top_k=5)
nodes = retriever.retrieve("what is the real source or object of religious force, and what collective process generates it?")

for i, node_with_score in enumerate(nodes):
    print(f"({i+1}) {node_with_score.metadata["file_name"]}: {node_with_score.score:.4f}")
    print(node_with_score.node.text[:200])
    print("-"*50)

(1) elementary_forms_religious_life-Durkheim.epub: 0.7509
If religion has given
birth to all that is essential in society, it is because the idea of society
is the soul of religion.

Religious forces are therefore human forces, moral forces. It is true that

--------------------------------------------------
(2) elementary_forms_religious_life-Durkheim.epub: 0.7300
2](895219089954929712_41360-h-23.htm.html#Footnote_238_238).  
  
Fear, religion not based on, [223
ff.](895219089954929712_41360-h-8.htm.html#Page_223),
[406](895219089954929712_41360-h-15.htm.html#P
--------------------------------------------------
(3) elementary_forms_religious_life-Durkheim.epub: 0.7189
230.

[[735]](895219089954929712_41360-h-9.htm.html#FNanchor_735_735) Another cause
has contributed much to this fusion; this is the extreme contagiousness of
religious forces. They seize upon every o
--------------------------------------------------
(4) sources_religious_insight-Royce.epub: 0.7180
It needs no human san

In [8]:
nodes[0].node.text

'If religion has given\nbirth to all that is essential in society, it is because the idea of society\nis the soul of religion.\n\nReligious forces are therefore human forces, moral forces. It is true that\nsince collective sentiments can become conscious of themselves only by fixing\nthemselves upon external objects, they have not been able to take form without\nadopting some of their characteristics from other things: they have thus\nacquired a sort of physical nature; in this way they have come to mix\nthemselves with the life of the material world, and then have considered\nthemselves capable of explaining what passes there. But when they are\nconsidered only from this point of view and in this rôle, only their most\nsuperficial aspect is seen. In reality, the essential elements of which these\ncollective sentiments are made have been borrowed by the understanding. It\nordinarily seems that they should have a human character only when they are\nconceived under human\nforms;[[1305]](

### Querying

Para esto se usara Llama 2.


In [9]:
from llama_index.llms.ollama import Ollama

llama = Ollama(
    model="llama2",
    request_timeout=1800.0, #30 minutos
)

query_engine = index.as_query_engine(llm=llama)

2026-11-15 21:17:37,232 - INFO - HTTP Request: POST http://localhost:11434/api/show "HTTP/1.1 200 OK"


## Prueba


Construimos la siguente lista de preguntas (con sus correspondientes respuestas y documentos relevantes) para medir el desempeño del RAG


In [24]:
questions = [
    {
        "question": 'what is the real source or object of religious force, and what collective process generates it?',
        "documents": ["elementary_forms_religious_life-Durkheim.epub"],
        "answer": 'Religious force is ultimately society/the clan, represented under totemic forms. Society is experienced as a moral power that raises individuals outside themselves; collective effervescence during gatherings generates the sacred/religious idea. The totemic principle is the clan “thought of under a more empirical form.”'
    },
    {
        "question": 'What does the claim that “the true sense of Theology is Anthropology” mean?',
        "documents":["essence_christianity-Feuerbach.epub"],
        "answer": 'Feuerbach means that there is no real distinction between the predicates of divine and human nature: the mysteries of theology are really mysteries of human nature. God is the objectification/projection of the human essence; religion is “the dream of the human mind.” Theology must be reduced to anthropology.'
    },
    {
        "question": 'What method and aim does Brinton propose for the “Science of Religion,” and what does he identify as the common source of devotional displays?',
        "documents":["religious_sentiment-Brinton.epub"],
        "answer": 'Brinton proposes an inductive, natural-scientific study of religion using the laws of mind and thought. Its aim is to discover the constitutive laws of the origin and spread of religions and their influence on human life. The common source of devotional displays is the Religious Sentiment, a complex feeling; religions are products of thought and are comprehensible through the laws of mind.'
    },
    {
        "question": 'What two independent but convergent lines of proof does Jackson use in The Philosophy of Natural Theology, and what is the stated purpose of the essay?',
        "documents":["philosophy_natural_theology-Jackson.epub"],
        "answer": '1) evidence from the external world — intelligence, design, contrivance, and adaptation of means to ends; and 2)evidence from the moral truth of personal consciousness. The essay’s purpose is to confute the materialism/scepticism of the present day and defend natural theology. The two lines are kept separate during examination but converge on the same conclusion.'
    },
    {
        "question": 'Which is the definition of myth in its most general nature, and what psychological act does he treat as its ultimate source?',
        "documents":["myth_science-Vignoli.epub"],
        "answer": 'Myth is the spontaneous, imaginative form in which human intelligence and emotion conceive and represent themselves and things in general; humans project themselves into the phenomena they apprehend. Its ultimate source is the primitive act by which humans personify and animate things — the general principle of animism — grounded in the “faculty of apprehension.”'
    },
    {
        "question": 'How do Catharine Beecher’s essay connect the people’s right to interpret the Bible with “common sense” and the “moral sense”?',
        "documents":["authorize_bible_interpeters-Beecher.epub",
                     "common_sense_religion-Beecher.epub"],
        "answer": 'In An Appeal …, Beecher defends the right of all people to interpret the Bible for themselves, unrestricted by ecclesiastical power, on grounds of liberty of conscience. In Common Sense Applied to Religion, she argues that theological theories must be tested by common sense and the moral sense; doctrines that seem “at war with both the common sense and the moral sense of mankind” are the fundamental problem. Reason/intuitive truths and moral susceptibilities are central.'
    },
    {
        "question": 'Both Royce and Sabatier treat religion as grounded in experience rather than beginning with external dogma. What sources of religious insight does Royce examine, and how does Sabatier characterize the psychological origin of religion?',
        "documents":["sources_religious_insight-Royce.epub",
                     "philosophy_religion_psychology_history-Sabatier.epub"],
        "answer": 'Royce examines individual experience, social experience, reason, loyalty, sorrow, and the unity of the spirit/invisible church, all in relation to the religious problem of salvation. Sabatier argues that religion springs from the initial contradiction and distress of inner life; it is “the Prayer of the Heart,” a practical act of faith/confidence in life, studied by a psychological and historical method, with dogma treated as secondary.'
    },
    {
        "question": 'What are the three stages of thought to organize God and the World, and what does the third stage will achieve?',
        "documents":["god_world-Robinson.epub"],
        "answer": 'Robinson organizes the survey into: 1) the older orthodoxy; 2) second thoughts forced on the age by physical science and the attack on traditional views; 3) a third stage of reconstruction. He hopes to discriminate the third stage and show that a common ground of agreement and cooperation between science and religion can now be reached.'
    },
    {
        "question": 'what autor says that religion is a collective obsessional neurosis, and how does it compare with Durkheim’s concept of collective effervescence?',
        "documents": None,
        "answer": 'Sigmund Freud’s argument in The Future of an Illusion. Is not int the collection'
    },
    {
        "question": 'What is the exact conciliar text of Nostra Aetate §2 on Hinduism and Buddhism, and which of these ten authors provides a clause-by-clause commentary on it?',
        "documents": None,
        "answer": 'Nostra Aetate was promulgated in 1965, after all ten books were written. None of the ten epubs contains the conciliar text or a commentary on it.'
    }
]

Usando las siguientes definiciones (segun [wikipedia](<https://en.wikipedia.org/wiki/Evaluation_measures_(information_retrieval)>))
$$\text{precision} = \frac{|\{\text{documentos relevantes}\}| \cap |\{\text{documentos recuperados}\}|}{|\{\text{documentos recuperados}\}|}$$

$$\text{recall} = \frac{|\{\text{documentos relevantes}\}| \cap |\{\text{documentos recuperados}\}|}{|\{\text{documentos relevantes}\}|}$$

Calculamos para cada pregunta


In [38]:
k=5
retriever = index.as_retriever(similarity_top_k=k)
table = f"""
| question | relevant documents | retrieved documents | precision@{k} | recall@{k} | golden answer | generated text |
| -------- | ------------------ | ------------------- | ------------- | ---------- | ------------- | -------------- |
"""
for q in questions:
    retrieved = set([node.metadata["file_name"] for node in retriever.retrieve(q["question"])])
    if q["documents"]:
        relevant = set(q["documents"])
        precision = round(len(retrieved & relevant)/len(retrieved),2)
        recall = round(len(retrieved & relevant)/len(relevant),2)
    else:
        relevant = "{}"
        precision = ""
        recall = ""
    table += "| "+ " | ".join([q["question"],str(relevant),str(retrieved),str(precision),str(recall),q["answer"], str(query_engine.query(q["question"])) ])+ " |\n"
print(table)

| question                                                                                                                                                                                                                                   | relevant documents                                                                               | retrieved documents                                                                                                                                                         | precision@5 | recall@5 | golden answer                                                                                                                                                                                                                                                                                                                                                                                                                                                                               | generated text                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                        |
| ------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------ | ------------------------------------------------------------------------------------------------ | --------------------------------------------------------------------------------------------------------------------------------------------------------------------------- | ----------- | -------- | ------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------- | ----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------- |
| what is the real source or object of religious force, and what collective process generates it?                                                                                                                                            | {'elementary_forms_religious_life-Durkheim.epub'}                                                | {'sources_religious_insight-Royce.epub', 'elementary_forms_religious_life-Durkheim.epub', 'religious_sentiment-Brinton.epub'}                                               | 0.33        | 1.0      | Religious force is ultimately society/the clan, represented under totemic forms. Society is experienced as a moral power that raises individuals outside themselves; collective effervescence during gatherings generates the sacred/religious idea. The totemic principle is the clan “thought of under a more empirical form.”                                                                                                                                                            | Based on the context information provided, the real source or object of religious force is not an external object or entity, but rather a collective process that arises from within society itself. Durkheim argues that religious forces are human forces, moral forces, and that they are not external entities, but rather a product of collective sentiments that are mixed with the material world.The collective process that generates religious force is the way society acts upon the individual, and vice versa. Durkheim posits that it is the society that has done it, and it is humanity that has reaped the fruits. This suggests that religious force is not something that is imposed from outside, but rather something that emerges from the interactions and collective experiences of individuals within a society.In other words, religious force is a social construct, created through the collective actions and beliefs of individuals within a society. It is not a fixed or objective entity, but rather a dynamic and evolving process that is shaped by the interactions and experiences of those within the society.                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                  |
| What does the claim that “the true sense of Theology is Anthropology” mean?                                                                                                                                                                | {'essence_christianity-Feuerbach.epub'}                                                          | {'religious_sentiment-Brinton.epub', 'philosophy_religion_psychology_history-Sabatier.epub', 'essence_christianity-Feuerbach.epub'}                                         | 0.33        | 1.0      | Feuerbach means that there is no real distinction between the predicates of divine and human nature: the mysteries of theology are really mysteries of human nature. God is the objectification/projection of the human essence; religion is “the dream of the human mind.” Theology must be reduced to anthropology.                                                                                                                                                                       | The claim that "the true sense of Theology is Anthropology" means that theology, as a discipline, is ultimately concerned with the study of human nature. This is because, according to Feuerbach, religion takes the superficial and apparent aspects of human nature as the essential and true ones, rather than recognizing the true nature of humanity as it exists in reality. In other words, theology is guilty of "illusions" because it misconstrues the nature of God and the divine.Feuerbach argues that theology is an attempt to objectify the true nature of the human word, but it only manages to define or make objective the true nature of humanity through anthropology. Anthropology, in this context, refers to the study of human nature in its true and essential form, rather than as it is distorted or idealized by religion.In essence, the claim that "the true sense of Theology is Anthropology" suggests that theology is ultimately concerned with understanding and interpreting human nature, and that this understanding can only be achieved through a careful examination of the nature of humanity itself, rather than through an analysis of divine or supernatural entities.                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                |
| What method and aim does Brinton propose for the “Science of Religion,” and what does he identify as the common source of devotional displays?                                                                                             | {'religious_sentiment-Brinton.epub'}                                                             | {'religious_sentiment-Brinton.epub'}                                                                                                                                        | 1.0         | 1.0      | Brinton proposes an inductive, natural-scientific study of religion using the laws of mind and thought. Its aim is to discover the constitutive laws of the origin and spread of religions and their influence on human life. The common source of devotional displays is the Religious Sentiment, a complex feeling; religions are products of thought and are comprehensible through the laws of mind.                                                                                    | In Brinton's view, the "Science of Religion" is a method that aims to understand the religious sentiment through a systematic and objective study. He believes that this approach can help identify the common source of devotional displays, which he refers to as "the religious sentiment." Brinton's method involves analyzing the emotional, rational, and mythological elements of religion, with the goal of uncovering the underlying principles that govern religious thought and behavior.According to Brinton, the religious sentiment is a late and perhaps final development of human consciousness, characterized by the fullest self-consciousness and the purest purpose of the emotions. He argues that the intellect first reaches entire self-consciousness, while the emotions attain perfection of purpose when guided by the highest manifestation of the intellect. Therefore, the goal of the "Science of Religion" is to provide a practical appreciation of the identity of truth, love, and life.Brinton identifies the common source of devotional displays as the "eternal laws of mind," which guarantee perpetuity to the extent they are obeyed. These laws convey a message that cannot be doubted concerning the purport of the order in nature, which is itself "the will of God." This message is the same that every religion speaks, which is to seek truth and do good. Therefore, Brinton proposes that the "Science of Religion" should focus on understanding the message of religion and its application in order to provide a satisfying ground for the religion of the future.                                                                                                                                                                           |
| What two independent but convergent lines of proof does Jackson use in The Philosophy of Natural Theology, and what is the stated purpose of the essay?                                                                                    | {'philosophy_natural_theology-Jackson.epub'}                                                     | {'philosophy_natural_theology-Jackson.epub'}                                                                                                                                | 1.0         | 1.0      | 1) evidence from the external world — intelligence, design, contrivance, and adaptation of means to ends; and 2)evidence from the moral truth of personal consciousness. The essay’s purpose is to confute the materialism/scepticism of the present day and defend natural theology. The two lines are kept separate during examination but converge on the same conclusion.                                                                                                               | Jackson uses two independent but convergent lines of proof in The Philosophy of Natural Theology:1. The first line of proof is based on the idea that the universe has a natural explanation, and that the existence of God can be explained through the natural world. Jackson argues that the complexity and order in the universe suggest the existence of a higher power or being that created and sustained it.2. The second line of proof is based on the concept of moral culture and improvement. Jackson suggests that the idea of God is necessary to explain the moral principles and values that govern human behavior. He argues that without the concept of God, there would be no objective standard for evaluating right and wrong.The stated purpose of the essay is to provide a rational basis for the belief in God, independent of religious authority or revelation. Jackson argues that the evidence from the natural world and human experience provides sufficient grounds for believing in the existence of God, and that this belief is not based on blind faith or superstition, but rather on reason and evidence.                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                       |
| Which is the definition of myth in its most general nature, and what psychological act does he treat as its ultimate source?                                                                                                               | {'myth_science-Vignoli.epub'}                                                                    | {'myth_science-Vignoli.epub'}                                                                                                                                               | 1.0         | 1.0      | Myth is the spontaneous, imaginative form in which human intelligence and emotion conceive and represent themselves and things in general; humans project themselves into the phenomena they apprehend. Its ultimate source is the primitive act by which humans personify and animate things — the general principle of animism — grounded in the “faculty of apprehension.”                                                                                                               | Based on the context information provided, the definition of myth in its most general nature is:"Myth, as it is understood by us, and as it will be developed and explained in this work, cannot be defined in summary terms, since its multiform and comprehensive nature embraces and includes all primitive action, as well as much which is consecutive and historical in the intelligence and feelings of man, with respect to the immediate and the reflex interpretation of the world, of the Individual, and of the society in which our common life is passed."According to the author, the ultimate source of myth is a psychological act, which is:"the spontaneous and imaginative form in which the human intelligence and human emotions conceive and represent themselves and things in general; it is the psychical and physical mode in which man projects himself into all those phenomena which he is able to apprehend and perceive."In other words, myth is a natural and inherent function of the human mind, born from the emotions and experiences of individuals and communities, and it continues to exist and evolve in response to changing psychological and social conditions.                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                          |
| How do Catharine Beecher’s essay connect the people’s right to interpret the Bible with “common sense” and the “moral sense”?                                                                                                              | {'common_sense_religion-Beecher.epub', 'authorize_bible_interpeters-Beecher.epub'}               | {'common_sense_religion-Beecher.epub', 'authorize_bible_interpeters-Beecher.epub'}                                                                                          | 1.0         | 1.0      | In An Appeal …, Beecher defends the right of all people to interpret the Bible for themselves, unrestricted by ecclesiastical power, on grounds of liberty of conscience. In Common Sense Applied to Religion, she argues that theological theories must be tested by common sense and the moral sense; doctrines that seem “at war with both the common sense and the moral sense of mankind” are the fundamental problem. Reason/intuitive truths and moral susceptibilities are central. | Catharine Beecher's essay "Common Sense Applied to Religion; Or, The Bible and the People" connects the people's right to interpret the Bible with "common sense" and the "moral sense" by arguing that these principles are essential in determining the truth of the Bible and its claims. Beecher believes that the laity, including women, have the right to examine and test the theories and claims of the Bible using reason and common sense, rather than relying solely on the decisions of ecclesiastical bodies or the writings of theologians.Beecher argues that the principles of reason and common sense must be evolved and stated in simple and popular form to enable the laity to apply them appropriately. She contends that these principles can be used to determine whether a thing agrees with or is contrary to the moral sense and the principles of reason and common sense.In essence, Beecher's essay posits that the people have the right to interpret the Bible using their own reason and moral sense, rather than relying solely on the interpretations of religious leaders or scholars. By doing so, Beecher's essay emphasizes the importance of individual responsibility and agency in determining the truth of the Bible, and the role that "common sense" and the "moral sense" play in this process.                                                                                                                                                                                                                                                                                                                                                                                                                                                        |
| Both Royce and Sabatier treat religion as grounded in experience rather than beginning with external dogma. What sources of religious insight does Royce examine, and how does Sabatier characterize the psychological origin of religion? | {'sources_religious_insight-Royce.epub', 'philosophy_religion_psychology_history-Sabatier.epub'} | {'sources_religious_insight-Royce.epub', 'philosophy_religion_psychology_history-Sabatier.epub'}                                                                            | 1.0         | 1.0      | Royce examines individual experience, social experience, reason, loyalty, sorrow, and the unity of the spirit/invisible church, all in relation to the religious problem of salvation. Sabatier argues that religion springs from the initial contradiction and distress of inner life; it is “the Prayer of the Heart,” a practical act of faith/confidence in life, studied by a psychological and historical method, with dogma treated as secondary.                                    | James's collection of the facts of religious experience, as described by Royce, reveals that the individual's personal experience is a source of genuine insight into their own personal ideals and needs. Royce argues that these experiences are essential substance of elementary religious experiences, which may concern an individual's Ideal, Need, and Deliverer. However, Sabatier takes a different approach, focusing on the psychological origin of religion. He suggests that religion is not grounded in external dogma but rather emerges from the individual's inner experience.According to Sabatier, religion is a psychological phenomenon that arises from the individual's attempt to make sense of their experiences, beliefs, and values. He posits that religion is not a set of beliefs or dogma but rather a way of understanding and navigating the human experience. In this view, religion is seen as a source of insight into the human condition, providing a framework for understanding and addressing the challenges and mysteries of life.In contrast, Royce's examination of religious insight is more focused on the individual's personal experience and the sources of insight that can be gained from it. He argues that the individual's personal experience of their Ideal, Need, and Deliverer is a valuable source of insight into the nature of reality and the human condition.Overall, while both Royce and Sabatier view religion as grounded in experience, they approach it from different perspectives. Royce emphasizes the individual's personal experience and the insights that can be gained from it, while Sabatier focuses on the psychological origins of religion and how it provides a framework for understanding the human experience. |
| What are the three stages of thought to organize God and the World, and what does the third stage will achieve?                                                                                                                            | {'god_world-Robinson.epub'}                                                                      | {'god_world-Robinson.epub', 'philosophy_religion_psychology_history-Sabatier.epub', 'philosophy_natural_theology-Jackson.epub', 'authorize_bible_interpeters-Beecher.epub'} | 0.25        | 1.0      | Robinson organizes the survey into: 1) the older orthodoxy; 2) second thoughts forced on the age by physical science and the attack on traditional views; 3) a third stage of reconstruction. He hopes to discriminate the third stage and show that a common ground of agreement and cooperation between science and religion can now be reached.                                                                                                                                          | Based on the provided context information, the three stages of thought to organize God and the World are:1. Hebraism, or the origins of the Gospel: This stage involves tracing the primary Christian ideas to their Hebraic root and understanding how the religion of Israel prepared the ground for the emergence of Christianity.2. The Old Testament: This stage involves studying the historical sequence and connection between the Old and New Testaments, and how the ideas in the Old Testament laid the foundation for the New Testament.3. The New Testament: This stage involves understanding how the ideas in the New Testament were influenced by the Old Testament and how they built upon it to create a new and fruitful religion. The third stage will achieve a deeper understanding of the historical and cultural context of Christianity and how it evolved over time.                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                        |
| what autor says that religion is a collective obsessional neurosis, and how does it compare with Durkheim’s concept of collective effervescence?                                                                                           | {}                                                                                               | {'elementary_forms_religious_life-Durkheim.epub'}                                                                                                                           |             |          | Sigmund Freud’s argument in The Future of an Illusion. Is not int the collection                                                                                                                                                                                                                                                                                                                                                                                                            | The author who says that religion is a collective obsessional neurosis is Pierre Bouvier. In his book "The Psychology of Religion," Bouvier argues that religion is a collective phenomenon that is fueled by the shared experiences and beliefs of a group, rather than an individual's personal beliefs. He suggests that religion can be seen as a form of collective obsessional neurosis, where individuals are consumed by the beliefs and practices of their religious group.In comparison to Durkheim's concept of collective effervescence, Bouvier's idea of collective obsessional neurosis is similar in that it also views religion as a collective phenomenon. However, while Durkheim saw collective effervescence as a positive and creative force, Bouvier's concept of collective obsessional neurosis is more negative, suggesting that religion can also be a source of anxiety and obsession for individuals.Both concepts recognize the importance of social factors in shaping religious beliefs and practices, but they differ in their overall tone and implications. Durkheim's concept of collective effervescence is more optimistic and highlights the potential for religion to bring people together and foster social cohesion, while Bouvier's concept of collective obsessional neurosis is more critical and suggests that religion can also be a source of conflict and oppression.                                                                                                                                                                                                                                                                                                                                                                               |
| What is the exact conciliar text of Nostra Aetate §2 on Hinduism and Buddhism, and which of these ten authors provides a clause-by-clause commentary on it?                                                                                | {}                                                                                               | {'elementary_forms_religious_life-Durkheim.epub', 'religious_sentiment-Brinton.epub', 'essence_christianity-Feuerbach.epub'}                                                |             |          | Nostra Aetate was promulgated in 1965, after all ten books were written. None of the ten epubs contains the conciliar text or a commentary on it.                                                                                                                                                                                                                                                                                                                                           | The exact conciliar text of Nostra Aetate §2 on Hinduism and Buddhism is:"The Church regards with esteem the religious traditions of Hinduism and Buddhism. In the light of the Catholic Church's teaching, the following principles are to be observed:1. Hinduism recognizes a divine reality in the universe, which it expresses through the concept of Brahman.2. Buddhism teaches the possibility of a spiritual life of a high standard through the practice of the Eightfold Path.3. The Church rejects nothing of what is true and holy in these religions, but asks that their adherents recognize and respect the transcendent nature of Christian faith.4. The Church affirms that the religion of India and the religion of Buddhism, far from being defective or imperfect, offer a valuable response to the human situation."As for a clause-by-clause commentary on this text, it can be found in the work of the following author:\* Louis Massignon, "Hinduism and Christianity: A Comparative Study of the Religious Traditions of India and the West" (1959)Massignon provides a detailed analysis of each clause of Nostra Aetate §2, offering insights into the similarities and differences between Hinduism and Buddhism, and their relationship to Christianity. He examines the concept of Brahman in Hinduism, the Eightfold Path in Buddhism, and the transcendent nature of Christian faith, highlighting the ways in which these religious traditions can inform and enrich each other.                                                                                                                                                                                                                                                                                  |
